# GIN Graph Model — pIC50 Prediction (ChEMBL)
Graph Isomorphism Network (GIN) on molecular graphs from SMILES.

In [1]:
# --- 0. Install PyTorch Geometric ---
import subprocess, sys

# Core PyG (CPU-only friendly)
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
#     "torch-geometric", "rdkit-pypi", "scikit-learn"])
# print("Done")

In [2]:
# --- 1. Imports ---
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader as PyGDataLoader
from torch_geometric.nn import GINConv, global_mean_pool, global_add_pool
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from rdkit import Chem
from rdkit.Chem import Descriptors
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

C:\Users\macie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [3]:
# --- 2. Config ---
SILVER_DIR   = os.path.join("chembl_work", "parquet_silver")
TARGETS_PATH = "targets_silver.parquet"   # in repo root (tid, organism only)
MAX_SAMPLES  = 10_000
BATCH_SIZE   = 64
LR           = 1e-3
NUM_EPOCHS   = 100
PATIENCE     = 15
GIN_HIDDEN   = 128
GIN_LAYERS   = 3
DROPOUT      = 0.3

In [4]:
# --- 3. Load data (same protein filter as MLP) ---
df_act = pd.read_parquet(os.path.join(SILVER_DIR, "activities_silver.parquet"))
df_ass = pd.read_parquet(os.path.join(SILVER_DIR, "assays_silver.parquet"))
df_mol = pd.read_parquet(os.path.join(SILVER_DIR, "molecules_silver.parquet"))

# Silver targets: only tid + organism (no pref_name)
df_tgt = pd.read_parquet(TARGETS_PATH)
df_tgt = df_tgt[df_tgt["organism"] == "Homo sapiens"].copy()
df_tgt["tid"] = df_tgt["tid"].astype(int)

df = df_act.merge(df_ass, on="assay_id", how="inner")
df = df.merge(df_tgt[["tid"]], on="tid", how="inner")
df = df.merge(df_mol, on="molregno", how="inner")
df = df.dropna(subset=["canonical_smiles", "pIC50"])

# Top protein by IC50 count (TID only)
top_protein_counts = df.groupby("tid").size().sort_values(ascending=False)
TOP_TID  = int(top_protein_counts.index[0])
TOP_NAME = f"TID_{TOP_TID}"
print(f"Protein: {TOP_NAME}  (count={top_protein_counts.iloc[0]:,})")

df_p = df[df["tid"] == TOP_TID].copy()
df_p = df_p[(df_p["pIC50"] >= 2) & (df_p["pIC50"] <= 12)]
if len(df_p) > MAX_SAMPLES:
    df_p = df_p.sample(n=MAX_SAMPLES, random_state=SEED)
df_p = df_p.reset_index(drop=True)
print(f"Samples: {len(df_p):,}")

Protein: TID_80224  (count=40,385)
Samples: 10,000


In [5]:
# --- 4. SMILES → molecular graph (PyG Data) ---

# Atom feature constants
ATOM_SYMBOLS = ['C','N','O','S','F','Si','P','Cl','Br','I','Na','K','Ca','Fe','B','other']
HYBRIDISATION = ['S','SP','SP2','SP3','SP3D','SP3D2','other']
DEGREE_MAX = 10
FORMAL_CHARGE_RANGE = list(range(-3, 4))   # -3 .. +3

def one_hot(val, lst):
    enc = [0] * len(lst)
    idx = lst.index(val) if val in lst else len(lst) - 1
    enc[idx] = 1
    return enc

def atom_features(atom) -> list:
    symbol = atom.GetSymbol() if atom.GetSymbol() in ATOM_SYMBOLS else 'other'
    hyb    = str(atom.GetHybridization()).split('.')[-1]
    if hyb not in HYBRIDISATION:
        hyb = 'other'
    charge = atom.GetFormalCharge()
    charge = max(FORMAL_CHARGE_RANGE[0], min(FORMAL_CHARGE_RANGE[-1], charge))
    feats = (
        one_hot(symbol, ATOM_SYMBOLS)       +   # 16
        one_hot(min(atom.GetDegree(), DEGREE_MAX), list(range(DEGREE_MAX+1)))  +  # 11
        one_hot(charge, FORMAL_CHARGE_RANGE) +   # 7
        one_hot(hyb, HYBRIDISATION)          +   # 7
        [int(atom.GetIsAromatic())]              # 1
    )
    return feats   # total: 42

NODE_FEAT_DIM = len(atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))
print(f"Node feature dimension: {NODE_FEAT_DIM}")

# Bond features
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def bond_features(bond) -> list:
    bt = bond.GetBondType()
    return (
        one_hot(bt, BOND_TYPES) +         # 4
        [int(bond.GetIsConjugated()),      # 1
         int(bond.IsInRing())]             # 1
    )   # total: 6

EDGE_FEAT_DIM = 6


def smiles_to_graph(smi: str, y: float):
    """Convert a SMILES string to a PyG Data object."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None

    # Node features
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)

    # Edge index + features (undirected: add both directions)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ef = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [ef, ef]

    if len(edge_index) == 0:
        return None

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)
    y_tensor   = torch.tensor([y],        dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)

Node feature dimension: 42


In [6]:
# --- 5. Build dataset ---
print("Converting SMILES to graphs...")
smiles_list = df_p["canonical_smiles"].tolist()
pic50_list  = df_p["pIC50"].tolist()

# Scale target
target_scaler = StandardScaler()
pic50_scaled  = target_scaler.fit_transform(
    np.array(pic50_list).reshape(-1, 1)).ravel()

graphs = []
for smi, y_raw, y_sc in zip(smiles_list, pic50_list, pic50_scaled):
    g = smiles_to_graph(smi, y_sc)
    if g is not None:
        g.y_orig = torch.tensor([y_raw], dtype=torch.float)
        graphs.append(g)

print(f"Valid graphs: {len(graphs):,} / {len(smiles_list):,}")

# Train / Val / Test split (indices)
idx = list(range(len(graphs)))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=SEED)
idx_val,   idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=SEED)

train_graphs = [graphs[i] for i in idx_train]
val_graphs   = [graphs[i] for i in idx_val]
test_graphs  = [graphs[i] for i in idx_test]

train_loader = PyGDataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = PyGDataLoader(val_graphs,   batch_size=BATCH_SIZE)
test_loader  = PyGDataLoader(test_graphs,  batch_size=BATCH_SIZE)

print(f"Train: {len(train_graphs)}  Val: {len(val_graphs)}  Test: {len(test_graphs)}")

Converting SMILES to graphs...
Valid graphs: 10,000 / 10,000
Train: 7000  Val: 1500  Test: 1500


In [7]:
# --- 6. GIN Model ---
class GINLayer(nn.Module):
    """Single GIN convolution layer.
    GIN update: h_v = MLP((1 + eps) * h_v + sum(h_u for u in N(v)))
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float = DROPOUT):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim * 2),
            nn.BatchNorm1d(out_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim * 2, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )
        # eps=0 and train_eps=True: learnable epsilon
        self.conv = GINConv(mlp, train_eps=True)

    def forward(self, x, edge_index):
        return self.conv(x, edge_index)


class GINModel(nn.Module):
    """
    Graph Isomorphism Network (GIN) for molecular property prediction.

    Architecture:
      Input projection -> N x GINLayer -> Global pooling -> MLP readout

    Pooling: concatenation of mean + sum pooling for richer graph representation.
    """
    def __init__(self, node_feat_dim: int, hidden_dim: int = GIN_HIDDEN,
                 n_layers: int = GIN_LAYERS, dropout: float = DROPOUT):
        super().__init__()

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(node_feat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )

        # GIN layers
        self.gin_layers = nn.ModuleList(
            [GINLayer(hidden_dim, hidden_dim, dropout) for _ in range(n_layers)]
        )

        # Readout MLP (concat mean + sum pooling -> 2 * hidden_dim)
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.input_proj(x)

        for layer in self.gin_layers:
            x = layer(x, edge_index)

        # Pooling: mean + sum
        x_mean = global_mean_pool(x, batch)
        x_sum  = global_add_pool(x, batch)
        x_graph = torch.cat([x_mean, x_sum], dim=-1)   # [N_graphs, 2*hidden]

        return self.readout(x_graph).squeeze(-1)


model_gin = GINModel(node_feat_dim=NODE_FEAT_DIM).to(DEVICE)
print(model_gin)
n_params = sum(p.numel() for p in model_gin.parameters())
print(f"\nTotal parameters: {n_params:,}")

GINModel(
  (input_proj): Sequential(
    (0): Linear(in_features=42, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (gin_layers): ModuleList(
    (0-2): 3 x GINLayer(
      (conv): GINConv(nn=Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.3, inplace=False)
        (4): Linear(in_features=256, out_features=128, bias=True)
        (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (6): ReLU()
      ))
    )
  )
  (readout): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=6

In [8]:
# --- 7. Training utilities ---
def gradient_norm_gin(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5


def evaluate_loader(model, loader):
    model.eval()
    preds_s, trues_s, trues_orig = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            out = model(batch).cpu().numpy()
            preds_s.extend(out)
            trues_s.extend(batch.y.cpu().numpy().ravel())
            trues_orig.extend(batch.y_orig.cpu().numpy().ravel())
    preds_s = np.array(preds_s)
    trues_s = np.array(trues_s)
    trues_o = np.array(trues_orig)
    # Inverse-scale predictions
    preds_orig = target_scaler.inverse_transform(preds_s.reshape(-1, 1)).ravel()
    loss = mean_squared_error(trues_s, preds_s)
    r2   = r2_score(trues_o, preds_orig)
    rmse = mean_squared_error(trues_o, preds_orig) ** 0.5
    return loss, r2, rmse, preds_orig, trues_o

In [ ]:
# --- 8. Training loop ---
optimizer = torch.optim.Adam(model_gin.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

history = {"train_loss": [], "val_loss": [], "val_r2": [], "grad_norm": [], "grad_delta": []}
best_val_loss = float("inf")
best_state    = None
patience_ctr  = 0
prev_g        = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model_gin.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model_gin(batch)
        loss = criterion(pred, batch.y.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model_gin.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item() * batch.num_graphs
    train_loss /= len(train_graphs)

    g_norm  = gradient_norm_gin(model_gin)
    g_delta = abs(g_norm - prev_g)
    prev_g  = g_norm

    val_loss, val_r2, val_rmse, _, _ = evaluate_loader(model_gin, val_loader)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_r2"].append(val_r2)
    history["grad_norm"].append(g_norm)
    history["grad_delta"].append(g_delta)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model_gin.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stop at epoch {epoch}")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Ep {epoch:03d} | train={train_loss:.4f}  val={val_loss:.4f}  "
              f"val_R²={val_r2:.4f}  |  grad={g_norm:.5f}  Δgrad={g_delta:.5f}")

print("\nLoading best weights...")
model_gin.load_state_dict(best_state)

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
# --- 9. Test evaluation ---
test_loss, test_r2, test_rmse, test_preds, test_true = evaluate_loader(model_gin, test_loader)

print("=" * 45)
print(f"TEST SET RESULTS  ({len(test_true):,} samples)")
print("=" * 45)
print(f"  MSE  : {test_loss:.4f}")
print(f"  RMSE : {test_rmse:.4f}")
print(f"  R²   : {test_r2:.4f}")
print("=" * 45)

In [ ]:
# --- 10. Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(history["train_loss"], label="Train")
ax.plot(history["val_loss"],   label="Val")
ax.set_title("GIN Loss"); ax.set_xlabel("Epoch"); ax.legend()

ax = axes[1]
ax.plot(history["val_r2"], color="green")
ax.axhline(test_r2, color="red", linestyle="--", label=f"Test R²={test_r2:.3f}")
ax.set_title("Val R²"); ax.set_xlabel("Epoch"); ax.legend()

ax = axes[2]
ax.plot(history["grad_norm"],  label="||grad||")
ax.plot(history["grad_delta"], label="Δ||grad||", alpha=0.7)
ax.set_title("Gradient norm"); ax.set_xlabel("Epoch"); ax.legend()

plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(test_true, test_preds, alpha=0.4, s=15)
mn, mx = min(test_true.min(), test_preds.min()), max(test_true.max(), test_preds.max())
plt.plot([mn, mx], [mn, mx], "r--", label="Perfect")
plt.xlabel("True pIC50"); plt.ylabel("Predicted pIC50")
plt.title(f"GIN: Actual vs Predicted  (R²={test_r2:.3f})")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- 11. Overfitting check ---
TINY_N = 200
tiny_train = train_graphs[:TINY_N]
tiny_loader2 = PyGDataLoader(tiny_train, batch_size=32, shuffle=True)

tiny_gin = GINModel(node_feat_dim=NODE_FEAT_DIM).to(DEVICE)
tiny_opt  = torch.optim.Adam(tiny_gin.parameters(), lr=LR)
tiny_crit = nn.MSELoss()

tt_losses, tv_losses = [], []
for epoch in range(1, 151):
    tiny_gin.train()
    for batch in tiny_loader2:
        batch = batch.to(DEVICE)
        tiny_opt.zero_grad()
        tiny_crit(tiny_gin(batch), batch.y.view(-1)).backward()
        tiny_opt.step()
    tiny_gin.eval()
    with torch.no_grad():
        tl_list = [tiny_crit(tiny_gin(b.to(DEVICE)), b.y.to(DEVICE).view(-1)).item() for b in tiny_loader2]
        tl = np.mean(tl_list)
        vl_list = [tiny_crit(tiny_gin(b.to(DEVICE)), b.y.to(DEVICE).view(-1)).item() for b in val_loader]
        vl = np.mean(vl_list)
    tt_losses.append(tl); tv_losses.append(vl)

plt.figure(figsize=(8, 4))
plt.plot(tt_losses, label=f"Train (n={TINY_N})")
plt.plot(tv_losses, label="Val (full)")
plt.title("GIN Overfitting check"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend()
plt.tight_layout(); plt.show()
gap = min(tv_losses) - min(tt_losses)
print(f"Train min: {min(tt_losses):.4f}  Val min: {min(tv_losses):.4f}  Gap: {gap:.4f}")